# 04_input_check — run before the GPU sessions

**Accelerator: None (CPU), Internet ON.** Attach exactly the inputs of a training session. Attach the competition, both cache datasets made from the 02 shard outputs (knee-cache-v3-shard0/1, latest version: 18 files of ~1 GB each), the four public report-label datasets, your reader_labels dataset and the 01_labels output (compi-label-generation, for the Flash-Lite labels). Inputs are detected automatically. Also attach the official RadImageNet ResNet-50 weights (marwanmath/resnet-50-radimagenet-marwan, file ResNet50.pt). It runs the training jobs' own data setup (cache shards, every label reader, gold rows), times random cache reads, and verifies the RadImageNet weights if attached. It uses no GPU quota; the last line must say INPUT CHECK PASSED.

In [ ]:
# Training requires Internet for pretrained weights and this small split helper.
import importlib.util, subprocess, sys
if importlib.util.find_spec('iterstrat') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','iterative-stratification==0.1.9'])
# Keep the preinstalled CUDA torch/torchvision stack; match inference's timm.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'timm==1.0.29'])

In [ ]:
import os
os.environ.setdefault("KNEE_ALLOW_CPU", "1")   # data setup only; nothing trains here

import os, gc, copy, math, time, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ---- inputs ------------------------------------------------------------------
"""Bounded discovery of attached Kaggle artifacts; embedded in launch notebooks."""
import fnmatch, os
from pathlib import Path


DICOM_DIRS = ('train_series', 'test_series')   # competition image trees: never listed


def attached_files(name, root='/kaggle/input', max_depth=5):
    # Kaggle mounts datasets and notebook outputs at different depths; artifacts live in
    # folders 1-5 levels below the root. The DICOM trees are pruned: listing them took
    # ~2 min on local disk and over an hour on the TPU VM's network-mounted inputs.
    root = Path(root)
    found = set()
    for path, dirs, files in os.walk(root):
        depth = len(Path(path).relative_to(root).parts)
        dirs[:] = [d for d in dirs if d not in DICOM_DIRS] if depth < max_depth else []
        if depth >= 1:
            found.update(Path(path) / f for f in fnmatch.filter(files, name) if not f.startswith('.'))
    return sorted(found)


def one_input(name, override='', root='/kaggle/input'):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f'Configured input does not exist: {path}')
        return str(path)
    matches = attached_files(name, root)
    if len(matches) != 1:
        raise RuntimeError(f'Attach exactly one intended {name} input or set its path explicitly. '
                           f'Found: {[str(p) for p in matches]}')
    print(f'{name}: {matches[0]}')
    return str(matches[0])


def fold_inputs(expected, root='/kaggle/input', per_fold=1):
    # per_fold > 1 when several training runs (e.g. DINOv2 and DINOv3 arms) are attached.
    paths = []
    for fold in expected:
        matches = attached_files(f'fold_{fold}.pt', root)
        if len(matches) != per_fold:
            raise RuntimeError(f'Attach exactly {per_fold} fold_{fold}.pt input(s) '
                               f'({"exactly one" if per_fold == 1 else "one per training run"}). '
                               f'Found: {[str(p) for p in matches]}')
        paths.extend(str(p) for p in matches)
    for path in paths: print(f'checkpoint: {path}')
    return paths


def checkpoint_inputs(root='/kaggle/input'):
    # Every attached fold_N.pt is an ensemble member; families are resolved from each
    # checkpoint's recipe, and duplicates of one family-fold are rejected by the caller.
    paths = [str(p) for p in attached_files('fold_*.pt', root)]
    if not paths:
        raise RuntimeError('Attach at least one training output containing fold_N.pt')
    for path in paths: print(f'checkpoint: {path}')
    return paths

# Cache shards and label tables are discovered under /kaggle/input. Set CACHE_DIR to a
# comma-separated list of shard folders only if the discovery is ambiguous.
INPUT_ROOT = os.environ.get("KNEE_INPUT_ROOT", "/kaggle/input")   # other hosts: where inputs live
CACHE_DIR = ','.join(str(p.parent) for p in attached_files('cache_meta.json', INPUT_ROOT))
assert CACHE_DIR, 'Attach the 02_cache shard outputs'
LABEL_ROOT = INPUT_ROOT
COMP_DIR  = os.environ.get("KNEE_COMP_DIR") or next(
    (d for d in ("/kaggle/input/competitions/rsna-knee-abnormality-detection", "/kaggle/input/rsna-knee-abnormality-detection")
     if os.path.isdir(d)), "/kaggle/input/competitions/rsna-knee-abnormality-detection")
OUT       = os.environ.get("KNEE_OUT", "/kaggle/working")   # one folder per family in a session
# -----------------------------------------------------------------------------
# Several cache shards may be attached; a comma-separated list is accepted too.
CACHE_DIRS = [d for d in CACHE_DIR.split(',') if d]

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

_metas = [json.load(open(os.path.join(d, 'cache_meta.json'))) for d in CACHE_DIRS]
_recipe_only = lambda m: {k: v for k, v in m.items() if k not in ('shard', 'n_shards')}
assert all(_recipe_only(m) == _recipe_only(_metas[0]) for m in _metas), 'Cache shards come from different recipes'
CACHE_META = _recipe_only(_metas[0])
_shards = sorted(m.get('shard', 0) for m in _metas)
assert _shards == list(range(_metas[0].get('n_shards', 1))), f'Attach every cache shard exactly once (found {_shards})'
assert CACHE_META.get('version') in ('knee-2c-v2', 'knee-2c-v3', 'knee-2c-v4', 'knee-2c-v5'), 'Rebuild the step-2c cache with current preprocessing'
PLANES = CACHE_META['slots']
CACHE_IMG = CACHE_META['img']
N_SLICES = CACHE_META['n_slices']

# ---- model recipe -------------------------------------------------------------
# Eight self-trained families, chosen by PRESET (one launcher set per family):
# reader_convnext: public 2.5D study reader (0.929 alone). ConvNeXt-T on windows of three
#   adjacent slices, slot and stack-position tokens, two transformer layers, attention
#   pooling per finding, EMA weights, warmup then cosine.
# reader_student: the ConvNeXt reader trained on report labels mixed with our teachers' OOF predictions.
# reader_student2: round 2, whose teacher also includes the round-1 students.
# reader_caformer: the same reader on CAFormer-S18 (ImageNet-22k), for backbone diversity.
# reader_coatnet: the same reader on CoAtNet-RMLP-2 at 384 px, the backbone of the public
#   Raptor single model (0.928 alone). Gradient checkpointing keeps it inside a T4.
# slot_dinov2: the public DINOv2 member (one spread 3-slice image per slot, last six
#   blocks fine-tuned, per-finding query head).
# radimagenet: frozen official RadImageNet ResNet-50 slice tokens with a per-finding query
#   head (public E9/E11 arm); encoded once, all folds in one run (rad_runtime.py).
# kneenet_b0: the earlier EfficientNet-b0 model, kept for comparison.
PRESET = os.environ.get("KNEE_PRESET", "reader_convnext")
PRESETS = {
    'reader_convnext': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                            epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                            pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                            clip=3., grad_checkpoint=False),
    # Noisy student: the same ConvNeXt reader recipe trained on targets mixed with the out-of-fold
    # predictions of our own readers (teacher_targets.csv). Agreement-gated: the teacher share runs
    # from 0.5 where the report readers agree confidently to 0.9 where they are split or silent.
    # Slot dropout hides whole MRI sequences in training (test sites differ in protocol).
    'reader_student': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                           epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                           pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                           clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # Round 2: the same student recipe, whose teacher also averages the round-1 students' OOF.
    # 11 epochs: pseudo-validation plateaus by epochs 10-12, and a two-fold session then fits
    # in under two GPU hours.
    'reader_student2': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                            epochs=11, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                            pct_start=1/11, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                            clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,
                            teacher_families={'reader_convnext': 1.0, 'reader_caformer': 1.0, 'reader_student': 1.0}),
    # Same reader, a different backbone family for ensemble diversity (convolution stages, then
    # attention; ImageNet-22k). ~23% more activation memory per image than ConvNeXt-T, whose
    # batch 4 peaked at 10.9 GB on a T4: half the micro-batch, same effective batch.
    'reader_caformer': dict(arch='reader25d', backbone='caformer_s18.sail_in22k_ft_in1k', img=CACHE_IMG,
                            epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                            pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                            clip=3., grad_checkpoint=False),
    # Round-1 student recipe and teacher on a ResNet-50 (batch-norm CNN) for ensemble diversity:
    # LB 2026-10-09/10: more ConvNeXt students/seeds added ~0.001; round-2 teachers did not help.
    # Half the micro-batch (activation memory), same effective batch.
    'reader_student_r50': dict(arch='reader25d', backbone='resnet50.a1_in1k', img=CACHE_IMG,
                               epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # Report-grounded student: the round-1 student recipe and teacher plus an auxiliary loss that
    # regresses each study's report embedding (scripts/report_embeddings.py; attach the
    # knee-report-embeddings dataset) from the images, at this weight relative to the label BCE.
    'reader_student_rep': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                               epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,
                               report_aux=.2),
    # Round-1 student recipe and teacher on CAFormer-S18 (in22k): its readers (session J) beat the
    # ConvNeXt readers on pseudo-validation (AUC 0.895/0.897 vs 0.893/0.891, folds 0/1); ResNet-50
    # students fell far behind (0.915 vs 0.963), so diversity comes from this family instead.
    'reader_student_caf': dict(arch='reader25d', backbone='caformer_s18.sail_in22k_ft_in1k', img=CACHE_IMG,
                               epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # Round-1 student recipe and teacher on ConvNeXt-Base (ImageNet-22k), ~3x ConvNeXt-T compute:
    # trained on the TPU (08_tpu_student_cnxb), where the long run does not hold up GPU sessions.
    'reader_student_cnxb': dict(arch='reader25d', backbone='convnext_base.fb_in22k_ft_in1k', img=CACHE_IMG,
                                epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # The round-1 ConvNeXt student recipe and teacher on the knee-2c-v4 cache (six v3 slots + two
    # 90 mm local slots around the header-geometry joint anchor); a separate family name so its
    # LB comparison with the v3 students (0.935 on folds 0-1) stays clean.
    'reader_student_v4': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                              epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                              pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                              clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # ConvNeXt-Base student on knee-2c-v4 (TPU): LB 2026-10-12 showed both gains separately,
    # ConvNeXt-B v3 x5 = 0.938 vs ConvNeXt-T v3 x6 = 0.936, and v4 folds 0-1 = 0.938 vs v3 = 0.935.
    'reader_student_cnxb_v4': dict(arch='reader25d', backbone='convnext_base.fb_in22k_ft_in1k', img=CACHE_IMG,
                                   epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                   pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                   clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # The ConvNeXt student on knee-2c-v5 (288 px, 10 windows per series; TPU).
    # 288 px x 8 slots: half the T4 micro-batch of v4 (activation memory), same effective batch of 8.
    'reader_student_v5': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                              epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                              pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                              clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # ConvNeXt-Base and CAFormer-S18 students on knee-2c-v5 (TPU / GPU after the 2026-10-17 quota reset).
    # On a T4: one study per micro-batch (48 windows at 288 px) with gradient checkpointing; the TPU
    # runtime ignores both (8 studies per step across the chips, no checkpointing). 8 epochs: at 50-60
    # min per epoch on a T4, 14 would exceed the 12 h session (2026-10-14); ConvNeXt-B at 288 px
    # reached val BCE 0.375 / AUC 0.954 by epoch 3.
    'reader_student_cnxb_v5': dict(arch='reader25d', backbone='convnext_base.fb_in22k_ft_in1k', img=CACHE_IMG,
                                   epochs=8, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                   pct_start=1/8, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                   clip=3., grad_checkpoint=True, teacher=(.5, .9), slot_dropout=.2),
    # v5 students taught partly by heliosli's compact model (its out-of-fold predictions on the training
    # studies, 10_compact_oof_*): teacher = our ConvNeXt readers (sessions A-F, the Z5/B5 teacher; no CAFormer
    # reader outputs are attached) and the compact model at 1.0 each, i.e. half ours and half compact, its
    # per-finding ranking put on our teachers' value distribution (its report targets use other label
    # definitions, e.g. synovitis mean 0.54 vs 0.30);
    # otherwise the ConvNeXt-T / -B v5 recipes.
    'reader_student_ct_v5': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                                 epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                 pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                 clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,
                                 teacher_families={'reader_convnext': 1.0, 'reader_compact': 1.0},
                                 teacher_quantile_map=('reader_compact',)),
    'reader_student_cnxb_ct_v5': dict(arch='reader25d', backbone='convnext_base.fb_in22k_ft_in1k', img=CACHE_IMG,
                                      epochs=8, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                      pct_start=1/8, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                      clip=3., grad_checkpoint=True, teacher=(.5, .9), slot_dropout=.2,
                                      teacher_families={'reader_convnext': 1.0, 'reader_compact': 1.0},
                                      teacher_quantile_map=('reader_compact',)),
    'reader_student_caf_v5': dict(arch='reader25d', backbone='caformer_s18.sail_in22k_ft_in1k', img=CACHE_IMG,
                                  epochs=7, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                  pct_start=1/7, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                  clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    'reader_coatnet': dict(arch='reader25d', backbone='coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k', img=384,
                           epochs=10, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                           pct_start=1/10, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                           clip=3., grad_checkpoint=True),
    'slot_dinov2': dict(arch='slot_query', backbone='vit_small_patch14_dinov2.lvd142m', img=224, epochs=10,
                        batch=4, accum=2, lr=1e-3, encoder_lr_scale=8e-3, wd=.02, pct_start=.15, dropout=.2,
                        unfreeze_last=6, head_hidden=256, ema=0., clip=5., grad_checkpoint=False),
    'radimagenet': dict(arch='radhead', backbone='radimagenet_resnet50', img=224, epochs=24, batch=48,
                        accum=1, lr=2e-4, encoder_lr_scale=0., wd=3e-3, pct_start=0., dropout=.15,
                        unfreeze_last=None, head_hidden=512, ema=0., clip=1., grad_checkpoint=False),
    'kneenet_b0': dict(arch='kneenet', backbone='tf_efficientnet_b0.ns_jft_in1k', img=CACHE_IMG, epochs=6,
                       batch=4, accum=1, lr=3e-4, encoder_lr_scale=.3, wd=1e-2, pct_start=.1, dropout=.4,
                       unfreeze_last=None, head_hidden=None, ema=0., clip=5., grad_checkpoint=False),
}
_preset = PRESETS[PRESET]
_needs = {'_v4': 'knee-2c-v4', '_v5': 'knee-2c-v5'}.get(PRESET[-3:])
assert _needs is None or CACHE_META.get('version') == _needs, f'Attach the {_needs} cache outputs'
ARCH = _preset['arch']
BACKBONE = os.environ.get("KNEE_BACKBONE", _preset['backbone'])
IMG = _preset['img']              # model input size; the cache is resized on the GPU if different
EPOCHS = _preset['epochs']
BATCH = _preset['batch']          # studies per micro-batch
ACCUM_STEPS = _preset['accum']    # effective batch = BATCH * ACCUM_STEPS studies
LR = _preset['lr']                # head learning rate
ENCODER_LR_SCALE = _preset['encoder_lr_scale']   # encoder rate = LR * this
WD = _preset['wd']
PCT_START = _preset['pct_start']  # warmup share of the one-cycle schedule
DROPOUT = _preset['dropout']
UNFREEZE_LAST = _preset['unfreeze_last']
HEAD_HIDDEN = _preset['head_hidden']
EMA_DECAY = _preset['ema']        # 0 disables; EMA weights are validated and saved
CLIP = _preset['clip']
GRAD_CHECKPOINT = _preset['grad_checkpoint']
TEACHER_WEIGHT = _preset.get('teacher', 0.0)   # constant or (lo, hi) gate; attach teacher_targets.csv
SLOT_DROPOUT = _preset.get('slot_dropout', 0.0) # reader25d: chance of hiding each present series in training
TEACHER_FAMILY_WEIGHTS = _preset.get('teacher_families')   # None: label_builder.TEACHER_FAMILIES
TEACHER_QUANTILE_MAP = _preset.get('teacher_quantile_map', ())   # outside teachers put on our teachers' scale
REPORT_AUX = _preset.get('report_aux', 0.0)    # weight of the report-embedding auxiliary loss; 0 disables
TRAIN_WINDOWS = 6                 # reader25d: windows encoded per series per step (of 8)
# Gold studies train in every fold at this per-cell weight (decision 2026-10-07). Their
# audit AUC is then in-sample and is printed as such; selection never reads gold.
GOLD_WEIGHT = 3.0
READER_WEIGHTS = {}               # e.g. {'extra_labels_sol_v2.csv': 2.0}; default 1 each
# Every reader the run must find; a missing attachment stops before any GPU time is spent.
REQUIRED_READERS = ['report_labels_v2.csv', 'llm_labels_v4_blend.csv', 'labels_llm_gpt56sol.csv',
                    'qwen_knee_weak_labels.csv', 'extra_labels_sol_v2.csv',
                    'extra_labels_sonnet_legacy.csv', 'pseudo_google_gemini*flash*.jsonl']
NORMALIZATION = 'pretrained'
N_FOLDS = 5
FOLDS_TO_RUN = json.loads(os.environ.get("KNEE_FOLDS", "[0,1,2,3,4]"))  # one fold first on Kaggle
GROUP_COL = 'report_group'        # folds keep duplicate reports together
RESUME = False  # copy both fold_N.pt and last_fold_N.pt to OUT before enabling
T_TRAIN     = 10       # kneenet only: contiguous slices sampled per plane per step
SEED        = int(os.environ.get("KNEE_SEED", 42))   # sessions may train extra seeds of a family
NUM_WORKERS = int(os.environ.get("KNEE_WORKERS", 4))   # Kaggle gives 4 cores; two-GPU sessions use 2 per job

def write_atomic(path, content):
    # Two jobs in one session may write the same small file (folds, config); replace it whole.
    # content: text, or a function that writes to the temporary path it is given.
    temp = f'{path}.{os.getpid()}.tmp'
    if callable(content):
        content(temp)
    else:
        with open(temp, 'w') as handle: handle.write(content)
    os.replace(temp, path)


os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.benchmark = True          # shapes are fixed -> let cuDNN autotune

print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")
if DEV == "cuda" and torch.cuda.device_count() > 1:
    print(f"note: {torch.cuda.device_count()} GPUs visible, using 1 by design "
          f"(DataParallel benchmarked 13x slower here)")
if ARCH == 'kneenet' and BACKBONE.startswith('vit_'):
    BATCH, ACCUM_STEPS = 1, 4  # effective batch 4; lower activation memory
print(f"recipe: {PRESET} ({ARCH}) | {BACKBONE} @ {IMG}px | {CACHE_META['version']} {len(PLANES)} slots x "
      f"{N_SLICES} slices x {CACHE_IMG}px in {len(CACHE_DIRS)} shard(s) | {EPOCHS} epochs, "
      f"{BATCH}x{ACCUM_STEPS} studies/step | gold weight {GOLD_WEIGHT}")
assert DEV == "cuda" or os.environ.get("KNEE_ALLOW_CPU"), "Enable a GPU before starting training"

RESUME_DIR = os.environ.get("KNEE_RESUME_DIR", "")
if RESUME_DIR:
    import shutil
    for fold in FOLDS_TO_RUN:
        for filename in (f'fold_{fold}.pt', f'last_fold_{fold}.pt'):
            previous = Path(RESUME_DIR) / os.environ.get('KNEE_RUN_NAME', PRESET) / filename
            if not previous.is_file():
                continue                       # this fold has no earlier progress
            destination = Path(OUT) / filename
            assert not destination.exists(), f'Restart the session before copying {destination}'
            shutil.copy2(previous, destination)
            RESUME = True


In [ ]:
"""Soft training targets from report-label readers; embedded in training.ipynb."""
import os, re, json, fnmatch, hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Public report-label tables (rules allow public external data). File name -> suffix of
# the twelve target columns. The mean of these four reached 0.896 macro AUC on the gold
# studies in the public 2.5D reader's own measurement.
PUBLIC_LABEL_TABLES = {
    'report_labels_v2.csv': '',            # pilkwang/rsna-knee-llm-labels
    'llm_labels_v4_blend.csv': '',         # stevenleehans/rsna-knee-llm-report-labels
    'labels_llm_gpt56sol.csv': '',         # lixin73/rsna-knee-llm-report-labels-sol56
    'qwen_knee_weak_labels.csv': '__label',  # laymond/rsna-knee-abnormality-qwen3-8b-weak-labels
}
# Our own readers:
#  - extra_labels_*.csv from scripts/export_reader_labels.py (GPT Sol v2, Sonnet legacy);
#  - the Flash member file written by our 01_labels notebook (its labels/ folder), read
#    directly so Flash counts once. That notebook's consensus train_targets.csv is not read:
#    it already mixes Flash with Sonnet.
OWN_PREFIX = 'extra_labels'
OWN_JSONL_READERS = {'pseudo_google_gemini*flash*.jsonl': 'legacy'}

# State -> probability that the finding is positive. Legacy is the 01_labels notebook's own
# HARD map; v2 states follow the knee-report-labels-v2 protocol. Unsure abstains (NaN)
# under legacy; under v2 it marks an affirmative finding of undecidable eligibility.
STATE_P = {
    'legacy': {'definite': 1.0, 'probable': .75, 'possible': .35, 'absent': 0.0},
    'v2': {'definite': 1.0, 'unsure': .5, 'absent': 0.0},
}
STATE_KEYS = dict(zip(
    ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
     "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"],
    ["acl", "mcl", "medial_meniscus", "lateral_meniscus", "medial_oa", "lateral_oa",
     "pf_oa", "effusion", "synovitis", "bakers", "contusion", "fracture"]))


def iter_jsonl(path):
    # One historical shard was written with literal "\n" separators; split on both.
    with open(path) as handle:
        for chunk in re.split(r'\n|\\n', handle.read()):
            if chunk.strip():
                yield json.loads(chunk)


def states_frame(records, label_cols, protocol):
    mapping = STATE_P[protocol]
    rows = [{'StudyInstanceUID': str(r['StudyInstanceUID']),
             **{c: mapping.get(str(r.get(STATE_KEYS.get(c, c), '')).lower()) for c in label_cols}}
            for r in records]
    frame = pd.DataFrame(rows, columns=['StudyInstanceUID', *label_cols])
    return frame.drop_duplicates('StudyInstanceUID', keep='last')


def find_label_tables(root='/kaggle/input'):
    found = {}
    for path, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        for name in files:
            if (name in PUBLIC_LABEL_TABLES or (name.startswith(OWN_PREFIX) and name.endswith('.csv'))
                    or any(fnmatch.fnmatch(name, pattern) for pattern in OWN_JSONL_READERS)):
                found.setdefault(name, []).append(os.path.join(path, name))
    duplicates = {k: v for k, v in found.items() if len(v) > 1}
    assert not duplicates, f'Attach each label table once: {duplicates}'
    flash = [k for k in found if any(fnmatch.fnmatch(k, p) for p in OWN_JSONL_READERS)]
    assert len(flash) <= 1, f'More than one Flash member file attached: {flash}'
    return {k: v[0] for k, v in sorted(found.items())}


def read_reader(path, label_cols):
    name = os.path.basename(path)
    if name.endswith('.jsonl'):
        protocol = next(v for p, v in OWN_JSONL_READERS.items() if fnmatch.fnmatch(name, p))
        frame = states_frame(iter_jsonl(path), label_cols, protocol)
        columns = list(label_cols)
    else:
        suffix = PUBLIC_LABEL_TABLES.get(name, '')
        frame = pd.read_csv(path, dtype={'StudyInstanceUID': str})
        columns = [c + suffix for c in label_cols]
    missing = [c for c in columns if c not in frame.columns]
    assert not missing, f'{name} lacks {missing[:3]}'
    assert not frame['StudyInstanceUID'].duplicated().any(), f'{name} repeats studies'
    values = frame.set_index('StudyInstanceUID')[columns].apply(pd.to_numeric, errors='coerce')
    values.columns = label_cols
    bad = (values < 0) | (values > 1)
    if bad.any().any():
        print(f'{name}: {int(bad.sum().sum())} values outside 0..1 treated as unobserved')
        values = values.mask(bad)
    return values


# Noisy-student targets (2026-10-08). teacher_targets.csv holds out-of-fold predictions of our own
# image models (scripts/teacher_targets.py); image models average out report noise, so their OOF
# predictions are cleaner targets than any single report reader. Gold rows never take teacher values.
TEACHER_FILE = 'teacher_targets.csv'


def find_teacher(root='/kaggle/input'):
    hits = []
    for path, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if TEACHER_FILE in files:
            hits.append(os.path.join(path, TEACHER_FILE))
    assert len(hits) <= 1, f'Attach one teacher table, found {hits}'
    return hits[0] if hits else None


# Family -> weight for teachers. Readers only: on the public LB the slot and RadImageNet families
# lowered the blend (17 models at equal weights 0.917 < two reader folds 0.923). Students and
# third-party checkpoints are never teachers.
TEACHER_FAMILIES = {'reader_convnext': 1.0, 'reader_caformer': 1.0}


def build_teacher(root, label_cols, gold_ids=(), families=None, quantile_map=()):
    """Teacher probabilities from the out-of-fold predictions (oof_fold_N.csv) of our own models
    found under root: seeds of a family are averaged, then families by weight. Every value is
    out-of-fold. Families in quantile_map (an outside model trained on other label definitions)
    keep their per-finding ranking but take the value distribution of the weighted mean of the
    other families before the average. Returns (frame with StudyInstanceUID, label_cols, n_members; meta)."""
    families = TEACHER_FAMILIES if families is None else families
    family_of = lambda folder: re.sub(r'_s\d+$', '', folder)
    members = {}
    for path, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        folder = os.path.basename(path)
        oofs = sorted(f for f in files if re.fullmatch(r'oof_fold_\d+\.csv', f))
        if oofs and family_of(folder) in families:
            members.setdefault(folder, []).extend(
                pd.read_csv(os.path.join(path, f), dtype={'StudyInstanceUID': str}) for f in oofs)
    assert members, f'no teacher OOF predictions under {root}'
    gold_ids, per_family = set(gold_ids), {}
    meta = dict(members={}, families={}, weights=dict(families))
    for folder, parts in sorted(members.items()):
        frame = pd.concat(parts, ignore_index=True)
        assert frame['StudyInstanceUID'].is_unique, f'{folder}: a study is out-of-fold twice'
        assert not gold_ids & set(frame['StudyInstanceUID']), f'{folder}: gold study in OOF'
        meta['members'][folder] = dict(studies=len(frame), folds=sorted(int(f) for f in frame['fold'].unique()))
        values = frame.set_index('StudyInstanceUID')[label_cols].astype(float)
        assert ((values >= 0) & (values <= 1)).all().all(), f'{folder}: OOF outside 0..1'
        per_family.setdefault(family_of(folder), []).append(values)
    ids = sorted(set().union(*[f.index for frames in per_family.values() for f in frames]))
    num = np.zeros((len(ids), len(label_cols))); den = np.zeros_like(num)
    n_members = np.zeros(len(ids), int)
    means = {}
    for family, frames in per_family.items():
        stack = np.stack([f.reindex(ids).to_numpy(float) for f in frames])
        counts = np.isfinite(stack).sum(axis=0)
        means[family] = (np.where(counts > 0, np.nansum(stack, axis=0) / np.maximum(counts, 1), np.nan), stack)
    mapped = [f for f in quantile_map if f in means]
    if mapped:
        ref_num = sum(np.nan_to_num(m) * families[f] for f, (m, _) in means.items() if f not in quantile_map)
        ref_den = sum(np.isfinite(m) * families[f] for f, (m, _) in means.items() if f not in quantile_map)
        assert np.max(ref_den) > 0, 'quantile_map needs at least one other teacher family'
        ref = np.where(ref_den > 0, ref_num / np.maximum(ref_den, 1e-9), np.nan)
        for family in mapped:
            m = means[family][0].copy()
            for j in range(m.shape[1]):
                ok, r = np.isfinite(m[:, j]), np.isfinite(ref[:, j])
                if ok.any() and r.any():
                    pct = (pd.Series(m[ok, j]).rank(method='average').to_numpy() - .5) / ok.sum()
                    m[ok, j] = np.quantile(ref[r, j], pct)
            means[family] = (m, means[family][1])
        meta['quantile_mapped'] = mapped
    for family, (mean, stack) in means.items():
        ok = np.isfinite(mean)
        num += np.where(ok, mean, 0) * families[family]
        den += ok * families[family]
        n_members += np.isfinite(stack).all(axis=2).sum(axis=0)
        meta['families'][family] = dict(members=len(frames), studies=int(ok.all(axis=1).sum()))
    teacher = pd.DataFrame(np.where(den > 0, num / np.maximum(den, 1e-9), np.nan), columns=label_cols)
    teacher.insert(0, 'StudyInstanceUID', ids)
    teacher['n_members'] = n_members
    return teacher, meta


def apply_teacher(frame, teacher_path, label_cols, weight):
    """Mix teacher probabilities into the pseudo rows: y = (1 - a) * report + a * teacher.
    weight is a constant a, or a gate (lo, hi): a = lo + (hi - lo) * u with u = 1 - |2 * report - 1|,
    so cells where the report readers agree confidently (consensus near 0 or 1) keep more of the
    report and split or uncertain cells lean on the image teacher. A pseudo cell without a report
    label takes the teacher value at cell weight hi. Gold rows are returned unchanged.
    Returns (frame, summary)."""
    lo, hi = (weight, weight) if np.isscalar(weight) else tuple(weight)
    assert 0 <= lo <= hi <= 1 and hi > 0
    teacher = pd.read_csv(teacher_path, dtype={'StudyInstanceUID': str})
    assert teacher['StudyInstanceUID'].is_unique, 'teacher repeats studies'
    t = teacher.set_index('StudyInstanceUID')[label_cols].apply(pd.to_numeric, errors='coerce')
    assert (t.isna() | ((t >= 0) & (t <= 1))).all().all(), 'teacher values outside 0..1'
    out = frame.copy()
    rows = np.flatnonzero(out['source'].eq('pseudo').to_numpy())
    wcols = [f'w_{c}' for c in label_cols]
    y = np.array(out.loc[out.index[rows], label_cols], dtype=float)
    w = np.array(out.loc[out.index[rows], wcols], dtype=float)
    tv = np.array(t.reindex(out['StudyInstanceUID'].iloc[rows]), dtype=float)
    has_t = np.isfinite(tv)
    both, only_t = has_t & np.isfinite(y), has_t & ~np.isfinite(y)
    a = lo + (hi - lo) * (1 - np.abs(2 * np.where(np.isfinite(y), y, .5) - 1))
    y[both] = (1 - a[both]) * y[both] + a[both] * tv[both]
    y[only_t] = tv[only_t]
    w[only_t] = hi
    out.loc[out.index[rows], label_cols] = np.where(np.isfinite(y), np.clip(y, .01, .99), np.nan).astype(np.float32)
    out.loc[out.index[rows], wcols] = w.astype(np.float32)
    summary = dict(teacher=teacher_path, weight=[lo, hi], mean_teacher_share=float(a[both].mean()) if both.any() else 0.,
                   studies=int(has_t.any(axis=1).sum()),
                   pseudo_studies=len(rows), mixed_cells=int(both.sum()), filled_cells=int(only_t.sum()),
                   sha256=hashlib.sha256(open(teacher_path, 'rb').read()).hexdigest())
    return out, summary


def report_groups(reports):
    # Byte-identical (up to case/punctuation) reports share targets: keep them in one fold.
    norm = reports.fillna('').astype(str).str.lower().map(lambda t: re.sub(r'\W+', '', t))
    return norm.map(lambda t: hashlib.sha1(t.encode()).hexdigest()[:16]).to_numpy()


def build_targets(train, label_cols, tables, reader_weights=None, gold_weight=3.0):
    """train: competition train.csv. Returns (frame, report) where frame holds soft
    targets, per-cell weights (w_<col>), source and report group per study."""
    ids = train['StudyInstanceUID'].astype(str)
    readers = {name: read_reader(path, label_cols).reindex(ids) for name, path in tables.items()}
    assert readers, 'Attach at least one report-label table'
    weights = {name: (reader_weights or {}).get(name, 1.0) for name in readers}
    num = sum(r[label_cols].fillna(0).to_numpy() * weights[n] for n, r in readers.items())
    den = sum(r[label_cols].notna().to_numpy() * weights[n] for n, r in readers.items())
    with np.errstate(invalid='ignore', divide='ignore'):
        soft = np.where(den > 0, num / np.maximum(den, 1e-9), np.nan)
    soft = np.clip(soft, .01, .99)
    gold = train[label_cols].apply(pd.to_numeric, errors='coerce')
    is_gold = gold.notna().all(axis=1).to_numpy()
    y = soft.astype(np.float32)
    w = np.where(np.isfinite(soft), 1.0, 0.0).astype(np.float32)
    y[is_gold] = gold.to_numpy(np.float32)[is_gold]
    w[is_gold] = gold_weight
    frame = pd.DataFrame(y, columns=label_cols)
    frame.insert(0, 'StudyInstanceUID', ids.to_numpy())
    for k, c in enumerate(label_cols): frame[f'w_{c}'] = w[:, k]
    frame['source'] = np.where(is_gold, 'gold', 'pseudo')
    frame['report_group'] = report_groups(train['Report']) if 'Report' in train else ids.to_numpy()
    # Diagnostic only: each reader against the image-read gold labels. Not used for any choice.
    report = {}
    for name, r in {**readers, 'consensus': pd.DataFrame(soft, index=ids, columns=label_cols)}.items():
        aucs = []
        # A reader that never labelled the gold studies has no gold AUC (not 0.5).
        gold_rows = int(r.loc[ids[is_gold], label_cols].notna().any(axis=1).sum())
        for c in label_cols:
            truth, score = gold.loc[is_gold, c].to_numpy(), r.loc[ids[is_gold], c].fillna(.5).to_numpy()
            aucs.append(roc_auc_score(truth, score) if len(np.unique(truth)) == 2 and gold_rows else np.nan)
        finite = [a for a in aucs if np.isfinite(a)]
        report[name] = dict(gold_macro_auc=float(np.mean(finite)) if finite else float('nan'),
                            gold_rows=gold_rows, weight=weights.get(name, 0),
                            coverage=float(np.isfinite(r.to_numpy()).mean()))
    return frame, report

# ---- cache shards -------------------------------------------------------------
# Rows are addressed as (shard, row in shard); every shard shares one recipe.
parts = []
for shard_id, directory in enumerate(CACHE_DIRS):
    part = pd.read_csv(os.path.join(directory, "cache_index.csv"))
    shape = np.load(os.path.join(directory, "cache.npy"), mmap_mode="r").shape
    assert shape[1:] == (len(PLANES), N_SLICES, CACHE_IMG, CACHE_IMG), f'{directory}: cache shape {shape}'
    assert len(part) == shape[0] and part['row'].between(0, shape[0] - 1).all()
    parts.append(part.assign(cache_id=shard_id))
    print(f"cache shard {shard_id}: {shape[0]:,} studies, {np.prod(shape)/1e9:.1f} GB  ({directory})")
index = pd.concat(parts, ignore_index=True)
assert index['StudyInstanceUID'].is_unique, 'a study appears in two cache shards'
T_TRAIN = min(T_TRAIN, N_SLICES)

# ---- targets: report-label readers + gold ------------------------------------
train_csv = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'), dtype={'StudyInstanceUID': str})
LABEL_TABLES = find_label_tables(LABEL_ROOT)
print('label readers:', list(LABEL_TABLES))
_missing = [r for r in REQUIRED_READERS if not any(fnmatch.fnmatch(name, r) for name in LABEL_TABLES)]
assert not _missing, f'Attach the label inputs providing {_missing} (see KAGGLE_RUN/README.md)'
targets, label_report = build_targets(train_csv, LABEL_COLS, LABEL_TABLES, READER_WEIGHTS, GOLD_WEIGHT)
print(pd.DataFrame(label_report).T.round(3).to_string())
print('(gold AUC of each reader is a diagnostic only; it chooses nothing. NaN: the reader has no gold rows)')
TEACHER_SUMMARY = None
if np.max(TEACHER_WEIGHT) > 0:
    _teacher = find_teacher(LABEL_ROOT)
    if _teacher is None:
        # No teacher table attached: build it here from the attached reader session outputs
        # (their out-of-fold predictions), exactly as 07_teacher_targets would.
        _gold = set(targets.loc[targets['source'].eq('gold'), 'StudyInstanceUID'])
        _frame, _meta = build_teacher(LABEL_ROOT, LABEL_COLS, gold_ids=_gold, families=globals().get('TEACHER_FAMILY_WEIGHTS'),
                                      quantile_map=globals().get('TEACHER_QUANTILE_MAP', ()))
        print('teacher built in session from:', json.dumps(_meta['members']))
        _absent = sorted(f for f, w in (globals().get('TEACHER_FAMILY_WEIGHTS') or {}).items() if w > 0 and f not in _meta['families'])
        assert not _absent, f'Attach the teacher inputs for families {_absent}'
        os.makedirs(OUT, exist_ok=True)
        _teacher = os.path.join(OUT, 'teacher_targets_used.csv')
        write_atomic(_teacher, lambda p: _frame.to_csv(p, index=False))
    targets, TEACHER_SUMMARY = apply_teacher(targets, _teacher, LABEL_COLS, TEACHER_WEIGHT)
    print('teacher targets:', TEACHER_SUMMARY)
write_atomic(os.path.join(OUT, 'targets_used.csv'), lambda p: targets.to_csv(p, index=False))

WEIGHT_COLS = [f'w_{c}' for c in LABEL_COLS]
df = index[index["n_present"] > 0].merge(targets, on="StudyInstanceUID", how="inner")
df = df[df[LABEL_COLS].notna().any(axis=1)].reset_index(drop=True)
print(f"\nusable studies: {len(df):,}  ({df['source'].value_counts().to_dict()})")
# Report-embedding auxiliary targets (training only): NaN rows have no embedding and are masked.
REP_COLS, AUX_DIM, REPORT_AUX_SUMMARY = [], 0, None
if globals().get('REPORT_AUX', 0.0) > 0:
    _rep_path = one_input('report_embeddings.csv', root=LABEL_ROOT)
    _rep = pd.read_csv(_rep_path, dtype={'StudyInstanceUID': str})
    REP_COLS = [c for c in _rep.columns if c.startswith('rep_')]
    assert REP_COLS and _rep['StudyInstanceUID'].is_unique and not set(REP_COLS) & set(df.columns)
    df = df.merge(_rep[['StudyInstanceUID', *REP_COLS]], on='StudyInstanceUID', how='left')
    AUX_DIM = len(REP_COLS)
    import hashlib as _hashlib
    REPORT_AUX_SUMMARY = dict(weight=REPORT_AUX, dims=AUX_DIM,
                              studies=int(df[REP_COLS[0]].notna().sum()),
                              sha256=_hashlib.sha256(open(_rep_path, 'rb').read()).hexdigest())
    print('report auxiliary targets:', REPORT_AUX_SUMMARY)
y_all = df[LABEL_COLS].to_numpy(np.float32)
print(f"observed label cells: {(~np.isnan(y_all)).mean():.1%}")
print("\nmean soft target (observed cells):")
print(pd.Series(np.nanmean(y_all[df['source'].eq('pseudo')], axis=0), index=LABEL_COLS).round(3).to_string())
assert df['source'].isin(['gold','pseudo']).all(), 'Unknown label source'
assert ((np.isnan(y_all)) | ((y_all>=0)&(y_all<=1))).all(), 'Targets outside [0,1]'
gold_check = train_csv.set_index('StudyInstanceUID')[LABEL_COLS].apply(pd.to_numeric, errors='coerce')
gold_ids = df.loc[df['source'].eq('gold'), 'StudyInstanceUID']
assert np.array_equal(df.loc[df['source'].eq('gold'), LABEL_COLS].to_numpy(float),
                      gold_check.loc[gold_ids].to_numpy(float)), 'Gold rows must carry the official labels'

In [ ]:
# ---- input check: what a GPU session will find ----------------------------------
import hashlib
RAD_SHA256 = "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
CHECK_STUDIES = 60
rng = np.random.default_rng(0)
mb_per_s = []
for d in CACHE_DIRS:
    arr = np.load(os.path.join(d, 'cache.npy'), mmap_mode='r')
    rows = rng.choice(arr.shape[0], size=min(CHECK_STUDIES, arr.shape[0]), replace=False)
    started, nbytes, blank = time.time(), 0, 0
    for r in rows:
        study = np.asarray(arr[r]); nbytes += study.nbytes; blank += int(study.max() == 0)
    mb_per_s.append(nbytes / 1e6 / max(time.time() - started, 1e-6))
    print(f'{d}: {len(rows)} random studies read at {mb_per_s[-1]:.0f} MB/s, {blank} blank')
epoch_gb = len(df) * np.prod(arr.shape[1:]) / 1e9
print(f'One reader epoch reads {epoch_gb:.1f} GB per job: about {epoch_gb * 1e3 / min(mb_per_s) / 60:.0f} min '
      'at this speed if nothing is cached (two jobs share the disk).')
weights = attached_files('ResNet50.pt', INPUT_ROOT)
if weights:
    digest = hashlib.sha256(open(weights[0], 'rb').read()).hexdigest()
    assert len(weights) == 1 and digest == RAD_SHA256, f'RadImageNet weights: {weights} {digest}'
    print('RadImageNet ResNet50.pt verified:', weights[0])
else:
    print('RadImageNet ResNet50.pt not attached (needed by session C only)')
print('\nINPUT CHECK PASSED: caches, label readers and gold rows are as training expects.')
